In [ ]:
'''CSV → data cleaning → type validation → join trades with market prices → valuation → P&L → exposure → risk'''

In [41]:
import pandas as pd
import numpy as np
from pathlib import Path


cwd = Path.cwd()
trade_file = cwd/ "energy_trading_data_example.csv"
df = pd.read_csv(trade_file)

market_file = cwd/ "energy_market_price_example.csv"
df_market = pd.read_csv(market_file)

In [44]:
df = to_numeric(df, ['volume','entry_price'])
df

,position_id,trade_timestamp,market,product,delivery_start,delivery_end,volume,unit,entry_price,price_unit
0,P001,10/7/2026 9:14,SE3,Day-Ahead,10/7/2026 18:00,19:00,20,MWh,85,EUR/MWh
1,P002,10/7/2026 9:18,DK1,Day-Ahead,10/7/2026 18:00,19:00,-20,MWh,175,EUR/MWh
2,P003,10/7/2026 9:32,DE-LU,Day-Ahead,10/7/2026 18:00,19:00,-15,MWh,180,EUR/MWh
3,P004,10/7/2026 10:05,SE3,Day-Ahead,10/7/2026 20:00,21:00,30,MWh,92,EUR/MWh
4,P005,10/7/2026 10:21,FI,Day-Ahead,10/8/2026 12:00,13:00,25,MWh,22,EUR/MWh
5,P006,10/7/2026 10:47,DK1,Day-Ahead,10/8/2026 19:00,20:00,-15,MWh,135,EUR/MWh
6,P007,10/7/2026 11:13,TTF,Month-Ahead,11/1/2026,12/1/2026,10,MW,31,EUR/MWh
7,P008,10/7/2026 11:41,DE-LU,Calendar-Year Future,1/1/2027,1/1/2028,-5,MW,82,EUR/MWh
8,P009,10/7/2026 12:03,SE3,Day-Ahead,10/8/2026 18:00,19:00,20,MWh,88,EUR/MWh
9,P010,10/7/2026 12:27,EUA,Dec-26 Future,12/1/2026,1/1/2027,1000,tCO₂,78,EUR/t


In [46]:
df_market = to_numeric(df_market, ['market_price'])
df_market

,market,product,delivery_start,market_price,price_unit,observed_at,source
0,SE3,Day-Ahead,10/7/2026 18:00,145,EUR/MWh,10/7/2026 15:00,Nord Pool
1,DK1,Day-Ahead,10/7/2026 18:00,205,EUR/MWh,10/7/2026 15:00,Nord Pool
2,DE-LU,Day-Ahead,10/7/2026 18:00,198,EUR/MWh,10/7/2026 15:00,EPEX
3,SE3,Day-Ahead,10/7/2026 20:00,132,EUR/MWh,10/7/2026 15:00,Nord Pool
4,FI,Day-Ahead,10/8/2026 12:00,18,EUR/MWh,10/7/2026 15:00,Nord Pool
5,DK1,Day-Ahead,10/8/2026 19:00,150,EUR/MWh,10/7/2026 15:00,Nord Pool
6,TTF,Month-Ahead,11/1/2026,33,EUR/MWh,10/7/2026 15:00,ICE
7,DE-LU,Calendar-Year Future,1/1/2027,79,EUR/MWh,10/7/2026 15:00,EEX
8,SE3,Day-Ahead,10/8/2026 18:00,110,EUR/MWh,10/7/2026 15:00,Nord Pool
9,EUA,Dec-26 Future,12/1/2026,80,EUR/t,10/7/2026 15:00,EEX


In [107]:
def to_numeric(df, columns):
    # to ensure numeric data
    df = df.copy()
    for col in columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("€", "", regex=False)
            .str.strip()
        )
        
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# errors="coerce" means that something like:
# "85.00"   → 85.0
# "20"      → 20.0
# "abc"     → NaN 


    return df

# def positions(df):
#     return df.groupby(['market','product','delivery_start','price_unit','unit'])['volume'].sum().reset_index().sort_values('volume', ascending=False)

def trades_market(df, df_market):
    return df.merge(df_market, on=['market','product','delivery_start', 'price_unit'], how='left')

def split_market(df_trades):
    df = df_trades.copy()

    market_mapping = {
        "SE3": ("Power", "SE3"),
        "DK1": ("Power", "DK1"),
        "DE-LU": ("Power", "DE-LU"),
        "FI": ("Power", "FI"),
        "TTF": ("Gas", "TTF"),
        "EUA": ("Carbon", "EUA"),
    }

    df["commodity"] = df["market"].map(
        lambda x: market_mapping[x][0]
    )

    df["zone_or_hub"] = df["market"].map(
        lambda x: market_mapping[x][1]
    )

    return df
    
def valuation(df_trades):
    # position_id, commodity, zone_or_hub, delivery, volume, entry_price, market_price, unrealized_p&l
    df = df_trades.copy()
    df['Unrealized_P&L'] = (df['market_price']-df['entry_price'])*df['volume']
    return df[['position_id', 'commodity', 'zone_or_hub', 'product','delivery_start', 'volume', 'unit', 'entry_price', 'market_price', 'price_unit', 'Unrealized_P&L']]

def exposure_power(df_value):
    df = df_value.copy()
    zone_exposure = (
        df
        .query("commodity == 'Power' and unit == 'MWh'")
        .groupby(['zone_or_hub'])['volume']
        .agg(
            net_position="sum",
            gross_position=lambda x: x.abs().sum()
        )
        .reset_index()
    )
    tot_exposure = zone_exposure['net_position'].sum()
    print(f"Net power exposure is {tot_exposure} MWh")
    return zone_exposure.sort_values('gross_position', ascending=False)

def exposure_hourly(df_value):
    df = df_value.copy()
    # df['delivery_start'] = pd.to_datetime(df['delivery_start'])
    hourly_exposure = (
        df.query("commodity == 'Power' and unit == 'MWh'")
        .groupby('delivery_start')
        .agg(
            net_position=("volume", "sum"),
            gross_position=("volume",
                            lambda x: x.abs().sum()),
            number_of_positions=("position_id", "count")
        )
        .reset_index()
        .sort_values('delivery_start')
    )
    return hourly_exposure


In [101]:
df_t = trades_market(df, df_market)
valuation(split_market(df_t))

,position_id,commodity,zone_or_hub,product,delivery_start,volume,unit,entry_price,market_price,price_unit,Unrealized_P&L
0,P001,Power,SE3,Day-Ahead,10/7/2026 18:00,20,MWh,85,145,EUR/MWh,1200
1,P002,Power,DK1,Day-Ahead,10/7/2026 18:00,-20,MWh,175,205,EUR/MWh,-600
2,P003,Power,DE-LU,Day-Ahead,10/7/2026 18:00,-15,MWh,180,198,EUR/MWh,-270
3,P004,Power,SE3,Day-Ahead,10/7/2026 20:00,30,MWh,92,132,EUR/MWh,1200
4,P005,Power,FI,Day-Ahead,10/8/2026 12:00,25,MWh,22,18,EUR/MWh,-100
5,P006,Power,DK1,Day-Ahead,10/8/2026 19:00,-15,MWh,135,150,EUR/MWh,-225
6,P007,Gas,TTF,Month-Ahead,11/1/2026,10,MW,31,33,EUR/MWh,20
7,P008,Power,DE-LU,Calendar-Year Future,1/1/2027,-5,MW,82,79,EUR/MWh,15
8,P009,Power,SE3,Day-Ahead,10/8/2026 18:00,20,MWh,88,110,EUR/MWh,440
9,P010,Carbon,EUA,Dec-26 Future,12/1/2026,1000,tCO₂,78,80,EUR/t,2000


In [102]:
df_val = valuation(split_market(df_t))
exposure_power(df_val)

Net power exposure is 45 MWh


,zone_or_hub,net_position,gross_position
3,SE3,70,70
1,DK1,-35,35
2,FI,25,25
0,DE-LU,-15,15


In [108]:
# analyse exposure by delivery hour
exposure_hourly(df_val)

,delivery_start,net_position,gross_position,number_of_positions
0,10/7/2026 18:00,-15,55,3
1,10/7/2026 20:00,30,30,1
2,10/8/2026 12:00,25,25,1
3,10/8/2026 18:00,20,20,1
4,10/8/2026 19:00,-15,15,1
